# BUEORM-DX fiel a los specs — Kaggle seguro

Este notebook instala **BUEORM-DX desde el repositorio GitHub publicado**, no reimplementa ni sustituye bloques. Usa el patrón completo `[BUEORM, BUEORM, Local, Relay]`, BPE subword, contexto 512 y streaming de RAM constante.

La recurrencia BUEORM es semánticamente secuencial. Este notebook mide el throughput antes de entrenar y no inicia una corrida extensa hasta que `RUN_TRAIN=True` se active expresamente. Así evita prometer un tiempo falso o gastar una sesión de Kaggle por accidente.

In [ ]:
# Kaggle: activa Internet y GPU (idealmente 2×T4) antes de ejecutar.
!pip -q install 'git+https://github.com/bueormnew/bda-lite.git@a04ec12' datasets


In [ ]:
import gc, json, math, random, time, zipfile
from dataclasses import asdict
from itertools import islice
from pathlib import Path

import torch
import torch.nn.functional as F
from datasets import load_dataset
from tokenizers import Tokenizer
from torch.utils.data import DataLoader, IterableDataset
from tqdm.auto import tqdm
from bda_lite import (BUEORMDX, BUEORMDXConfig, BUEORMCoreConfig, LocalAttentionConfig, GlobalRelayConfig, train_bpe)

SEED=42; random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Activa GPU en Kaggle.'
N_GPU=torch.cuda.device_count(); print('GPUs:', [torch.cuda.get_device_name(i) for i in range(N_GPU)])
DEVICE=torch.device('cuda:0'); OUT=Path('/kaggle/working/bueorm_dx_es'); OUT.mkdir(parents=True, exist_ok=True)
CONTEXT=512; VOCAB_SIZE=4096; BATCH_PER_GPU=1; GRAD_ACCUM=4
# Seguridad: primero ejecuta preflight. Cambia a True sólo si aceptas el ETA impreso.
RUN_TRAIN=False; TRAIN_TOKENS=500_000; PREFLIGHT_STEPS=8; MAX_SESSION_HOURS=8
torch.backends.cudnn.benchmark=True


In [ ]:
# BPE pequeño, no byte-level. Preparación en dos fases visibles y acotadas.
# La lista BPE usa como máximo 512 × 2048 caracteres (~1 MiB), nunca el corpus entero.
DATASET='SINAI/ALIA-cultural-heritage'; TEXT='text'; BPE_DOCS=512; HOLDOUT_DOCS=128; MAX_BPE_CHARS=2048; MAX_ENCODE_CHARS=8192
def text_stream(skip=0):
    for row in load_dataset(DATASET, split='train', streaming=True).skip(skip):
        yield row[TEXT]

# Fase 1: si se detiene aquí, es la red/dataset; NO es el entrenamiento del tokenizer.
print('Fase 1/2 — obteniendo muestra BPE desde Hugging Face...')
bpe_texts=[]
for text in tqdm(islice(text_stream(), BPE_DOCS), total=BPE_DOCS, desc='Descargando muestra BPE', unit='docs', dynamic_ncols=True):
    if isinstance(text,str) and text.strip(): bpe_texts.append(text[:MAX_BPE_CHARS])
if len(bpe_texts)<32: raise RuntimeError('No llegaron suficientes textos para entrenar BPE. Comprueba Internet y el acceso al dataset.')
# Fase 2: iterable local y finito: tokenizers no toca red y termina rápidamente.
print(f'Fase 2/2 — entrenando BPE local con {len(bpe_texts)} textos ({sum(map(len,bpe_texts)):,} caracteres)...')
tokenizer=train_bpe(bpe_texts, OUT/'tokenizer.json', VOCAB_SIZE)
print('BPE terminado. Vocabulario:', tokenizer.get_vocab_size())
del bpe_texts; gc.collect()

class Blocks(IterableDataset):
    # Cada bloque se crea, entrena y libera: no tokeniza ni retiene el corpus entero.
    def __init__(self, skip=0): super().__init__(); self.skip=skip
    def __iter__(self):
        bos,eos=tokenizer.token_to_id('<bos>'),tokenizer.token_to_id('<eos>'); pending=[]
        for text in text_stream(self.skip):
            for start in range(0,len(text),MAX_ENCODE_CHARS):
                pending.extend([bos,*tokenizer.encode(text[start:start+MAX_ENCODE_CHARS]).ids,eos])
                while len(pending)>=CONTEXT+1:
                    yield torch.tensor(pending[:CONTEXT+1],dtype=torch.long)
                    del pending[:CONTEXT+1]

GLOBAL_BATCH=max(1,N_GPU)*BATCH_PER_GPU
train_loader=DataLoader(Blocks(skip=HOLDOUT_DOCS),batch_size=GLOBAL_BATCH,num_workers=0,pin_memory=True)
val_loader=DataLoader(Blocks(),batch_size=GLOBAL_BATCH,num_workers=0,pin_memory=True)


In [ ]:
# Arquitectura literal del spec: BUEORM ×2, atención local y Global Latent Relay.
# Se reduce ancho/vocabulario, no se eliminan mecanismos arquitectónicos.
cfg=BUEORMDXConfig(
    vocab_size=tokenizer.get_vocab_size(), d_model=128, n_macrocycles=1,
    block_pattern=('bueorm','bueorm','local','relay'),
    core=BUEORMCoreConfig(128,n_heads=4,n_anchors=8,beta_max=.15,lam=.99,spectral_budget=4.,thermostat_every=64,conv_kernel=4,half_life_min=8,half_life_max=1024),
    local=LocalAttentionConfig(128,n_heads=4,window=64,rope_frac=.125),
    relay=GlobalRelayConfig(128,chunk_size=64,latent=64,n_queries=2,summary_layers=1,summary_heads=4,recency_window=4),
)
raw_model=BUEORMDX(cfg).to(DEVICE)
print(raw_model.block_summary())
print(f'Parámetros: {sum(p.numel() for p in raw_model.parameters()):,}')
# DataParallel usa ambas T4; batch global=2 conserva una secuencia por GPU.
model=torch.nn.DataParallel(raw_model) if N_GPU>1 else raw_model
optimizer=torch.optim.AdamW(model.parameters(),lr=3e-4,betas=(.9,.95),weight_decay=.1)
scaler=torch.amp.GradScaler('cuda')
torch.cuda.empty_cache()


In [ ]:
# Preflight: benchmark real. Nunca calcula ETA por documentos ni presupone throughput.
def step(batch, backward=True):
    batch=batch.to(DEVICE,non_blocking=True); x,y=batch[:,:-1],batch[:,1:]
    with torch.autocast('cuda',dtype=torch.float16):
        logits,_=model(x); loss=F.cross_entropy(logits.reshape(-1,cfg.vocab_size),y.reshape(-1))
    if backward: scaler.scale(loss/GRAD_ACCUM).backward()
    return loss.detach(),x.numel()

optimizer.zero_grad(set_to_none=True); torch.cuda.reset_peak_memory_stats()
seen=0; started=time.perf_counter()
for i,batch in enumerate(tqdm(islice(train_loader,PREFLIGHT_STEPS),total=PREFLIGHT_STEPS,desc='Preflight BUEORM-DX',unit='steps')):
    loss,n=step(batch); seen+=n
    if (i+1)%GRAD_ACCUM==0:
        scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
elapsed=time.perf_counter()-started; tok_s=seen/elapsed; estimate=TRAIN_TOKENS/tok_s/3600
print(f'preflight_loss={loss.item():.3f}; throughput={tok_s:.1f} tok/s; peak_vram={torch.cuda.max_memory_allocated()/2**30:.2f} GB; estimación para {TRAIN_TOKENS:,} tokens={estimate:.2f} h')
assert math.isfinite(loss.item()) and tok_s>0
if RUN_TRAIN and estimate>MAX_SESSION_HOURS: raise RuntimeError(f'Abortado: ETA {estimate:.1f}h > límite {MAX_SESSION_HOURS}h. Baja TRAIN_TOKENS o implementa kernel WY/Triton.')
if not RUN_TRAIN: print('Preflight correcto. Revisa el ETA y cambia RUN_TRAIN=True para iniciar una corrida real.')


In [ ]:
# Entrenamiento por presupuesto de tokens, con checkpoints portables.
# Ejecuta esta celda sólo después de activar RUN_TRAIN=True y volver a ejecutar las celdas previas.
if not RUN_TRAIN: raise RuntimeError('Protección activa: define RUN_TRAIN=True tras validar el preflight.')
def save(tag, step_count, tokens):
    payload={'model':raw_model.state_dict(),'optimizer':optimizer.state_dict(),'scaler':scaler.state_dict(),'step':step_count,'tokens':tokens,'config':asdict(cfg)}
    torch.save(payload,OUT/f'{tag}.pt')

raw_model.train(); optimizer.zero_grad(set_to_none=True); tokens=0; steps=0; started=time.perf_counter(); CHECKPOINT_EVERY=100
bar=tqdm(total=TRAIN_TOKENS,desc='Entrenando BUEORM-DX',unit='tokens',dynamic_ncols=True)
for batch in train_loader:
    loss,n=step(batch); tokens+=n; steps+=1
    if steps%GRAD_ACCUM==0:
        scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
    if steps%CHECKPOINT_EVERY==0: save('latest',steps,tokens)
    bar.update(n); bar.set_postfix(loss=f'{loss.item():.3f}',tok_s=f'{tokens/max(time.perf_counter()-started,1e-6):.1f}')
    if tokens>=TRAIN_TOKENS: break
if steps%GRAD_ACCUM: scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(optimizer); scaler.update()
bar.close(); save('final',steps,tokens); print(f'Final: {tokens:,} tokens, {tokens/(time.perf_counter()-started):.1f} tok/s')


In [ ]:
# Validación breve, empaquetado y prueba de carga local. No requiere dataset para reabrir el bundle.
@torch.inference_mode()
def validate(max_batches=8):
    raw_model.eval(); values=[]
    for batch in tqdm(islice(val_loader,max_batches),total=max_batches,desc='Validación',unit='batches'):
        batch=batch.to(DEVICE); logits,_=raw_model(batch[:,:-1]); values.append(F.cross_entropy(logits.reshape(-1,cfg.vocab_size),batch[:,1:].reshape(-1)).item())
    return sum(values)/len(values)
val_loss=validate(); manifest={'architecture':'BUEORM-DX','source':'https://github.com/bueormnew/bda-lite','context_length':CONTEXT,'parameter_count':sum(p.numel() for p in raw_model.parameters()),'validation_loss':val_loss,'config':asdict(cfg)}
(OUT/'config.json').write_text(json.dumps(manifest,indent=2)); tokenizer.save(str(OUT/'tokenizer.json'))
with zipfile.ZipFile(OUT/'bueorm_dx_bundle.zip','w',zipfile.ZIP_DEFLATED) as archive:
    for path in tqdm([OUT/'final.pt',OUT/'config.json',OUT/'tokenizer.json'],desc='Empaquetando',unit='files'): archive.write(path,path.name)
print(f'validation_loss={val_loss:.3f}; bundle={OUT/"bueorm_dx_bundle.zip"}')
